In [1]:
import torch
from torch.utils.data import Dataset

PAD_ID = 2

def flatten_state(cur_seq):
    flat = []
    for seq in cur_seq:
        for b in seq:
            flat.append(1 if b > 0 else 0)
    return torch.tensor(flat, dtype=torch.long)

class PrefixValuePaddedDataset(Dataset):
    def __init__(self, samples):
        self.data = []
        for seq, label in samples:
            x = flatten_state(seq)
            self.data.append((x, label))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

In [2]:
def collate_pad(batch):
    """
    batch: list of (x, y) where x is [T_i] of 0/1 tokens
    returns:
      X: [B, T] int64 in {0,1,PAD_ID}
      Y: [B]     int64 in {0,1}
      M: [B, T]  bool   True where PAD
    """
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    padded, masks = [], []
    for x in xs:
        pad_len = max_len - x.size(0)
        if pad_len > 0:
            x_pad = torch.cat([x, torch.full((pad_len,), PAD_ID, dtype=torch.long)])
            m_pad = torch.cat([torch.zeros(x.size(0), dtype=torch.bool),
                               torch.ones(pad_len, dtype=torch.bool)])   # True = PAD
        else:
            x_pad = x
            m_pad = torch.zeros_like(x, dtype=torch.bool)
        padded.append(x_pad); masks.append(m_pad)

    X = torch.stack(padded)                   # [B, T]
    Y = torch.tensor(ys, dtype=torch.float)    # [B]
    M = torch.stack(masks)                    # [B, T] (True=PAD)
    return X, Y, M

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ValueOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=3,       # tokens {0,1,PAD_ID}
        pad_id=PAD_ID,
        max_len=4096,
        d_model=256,
        nhead=8,
        nlayers=4,
        dim_ff=512,
        dropout=0.1,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb   = nn.Embedding(max_len, d_model)  # learned positional embeddings

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, 1)  # single logit for feasibility

        # (optional) used if a row is all PAD to avoid NaNs
        self.empty_embed = nn.Parameter(torch.zeros(d_model))

    def forward(self, x, pad_mask):
        """
        x:        [B, T] int64 in {0,1,pad_id}
        pad_mask: [B, T] bool, True where PAD (ignored by attention)
        returns:  [B] raw logits (apply sigmoid for probs)
        """
        B, T = x.size()

        # identify non-empty rows (at least one non-pad token)
        valid = ~pad_mask                         # True where real token
        nonempty = valid.any(dim=1)               # [B]
        empty = ~nonempty                         # [B]

        # container for pooled representations
        pooled = x.new_zeros((B, self.token_emb.embedding_dim), dtype=torch.float)

        if nonempty.any():
            x_ne = x[nonempty]                    # [B_ne, T]
            m_ne = pad_mask[nonempty]             # [B_ne, T]
            B_ne = x_ne.size(0)

            pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B_ne, T)  # [B_ne, T]
            h = self.token_emb(x_ne) + self.pos_emb(pos)                          # [B_ne, T, D]
            h = self.encoder(h, src_key_padding_mask=m_ne)                        # [B_ne, T, D]

            v_ne = ~m_ne
            lengths = v_ne.long().sum(dim=1).clamp(min=1) - 1                     # [B_ne]
            idx = lengths.view(B_ne, 1, 1).expand(B_ne, 1, h.size(-1))
            pooled_ne = h.gather(1, idx).squeeze(1)                               # [B_ne, D]
            pooled[nonempty] = pooled_ne

        if empty.any():
            pooled[empty] = self.empty_embed

        logit = self.head(self.norm(pooled)).squeeze(-1)                           # [B]
        return logit

In [4]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch

def train_overfit(samples, epochs=50, batch_size=32, device=None, pos_weight=None):
    """
    samples: list of ([(A...), (B...), (C...), (D...)], value_float_in_[0,1])
    Expects your Value dataset class to pad & map to (X,Y,M) with Y float.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # Build dataset & loader (use your own Value dataset class)
    ds = PrefixValuePaddedDataset(samples)                      # -> (X [T_i], y float, M [T_i])
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, collate_fn=collate_pad)

    # Model
    model = ValueOnlyTransformer(vocab_size=3, pad_id=PAD_ID, max_len=4096).to(device)

    # Loss (handle class imbalance if needed)
    # if pos_weight is None:
    #     with torch.no_grad():
    #         p = ds.y.mean().item() if len(ds) else 0.5   # estimate positive rate
    #     pos_weight = torch.tensor([(1 - p) / max(p, 1e-6)], device=device)
    # else:
    #     pos_weight = torch.tensor([pos_weight], device=device)

    #criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    criterion = nn.BCEWithLogitsLoss()
    opt = optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01)

    for ep in range(1, epochs + 1):
        model.train()
        tot_loss = tot_acc = n = 0
        for X, Y, M in loader:                           # X:[B,T] int {0,1,PAD}, Y:[B] float, M:[B,T] bool
            X, Y, M = X.to(device), Y.to(device), M.to(device)

            logit = model(X, pad_mask=M)                 # [B]
            loss = criterion(logit, Y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            with torch.no_grad():
                prob = torch.sigmoid(logit)              # [B]
                pred = (prob >= 0.5).float()
                acc  = (pred == Y.round()).float().sum().item()  # if Y is soft, this still treats >=.5 as pos
            bs = X.size(0)
            tot_loss += loss.item() * bs
            tot_acc  += acc
            n += bs

        if ep % 1 == 0:
            print(f"epoch {ep:3d} | loss {tot_loss/n:.4f} | acc {tot_acc/n:.3f}")

    return model

In [1]:
from sequence_generator import generate_turyn_values, read_seq, write_seq

## Create dataset on specified length and evaluate on that length

r = 9
sat_file = "sat_prefixes_9.txt"
unsat_file = "unsat_prefixes_9.txt"

generate_turyn_values(r, sat_file, unsat_file)
# sats = read_seq(sat_file)
# unsats = read_seq(unsat_file)
# original_dataset = []
# for item in sats:
#     original_dataset.append((item, 1))
# for item in unsats:
#     original_dataset.append((item, 0))

# print("Started training")
# model = train_overfit(original_dataset, epochs=20)

In [11]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

@torch.no_grad()
def evaluate_value(model, dataset, batch_size=256, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_pad)

    tot_loss = n = 0
    tot_correct = 0
    for X, Y, M in loader:
        X, Y, M = X.to(device), Y.to(device).float(), M.to(device)   # cast Y -> float
        logit = model(X, pad_mask=M)                                 # [B]
        loss  = F.binary_cross_entropy_with_logits(logit, Y, reduction="sum")

        prob = torch.sigmoid(logit)
        pred = (prob >= 0.5).float()
        tot_correct += (pred == (Y >= 0.5)).float().sum().item()

        tot_loss += loss.item()
        n += X.size(0)

    return {
        "loss": tot_loss / max(n, 1),
        "acc":  tot_correct / max(n, 1),   # thresholded accuracy
        "n": n,
    }

In [46]:
import random

single_example = unsats[int(random.random() * len(sats))]
print(single_example)

single_eval_dataset = PrefixValuePaddedDataset([(single_example, 0)])
evaluate_value(model, single_eval_dataset)

[(1, 1, 1, -1, 1, -1), (1, -1, 1, -1, -1, 1), (1, -1, 1, -1, -1, 1), (1, 1, 1, -1, 1)]
tensor([0.6856])


{'loss': 1.1570074558258057, 'acc': 0.0, 'n': 1}

In [13]:
eval_dataset = PrefixValuePaddedDataset(original_dataset)
print("Length:", r)
print(evaluate_value(model, eval_dataset))

Length: 7
{'loss': 0.13036151507927657, 'acc': 0.9433681073025335, 'n': 1342}


In [53]:
from sequence_generator import create_partial_str

strs = set()

for item in sats:
    strs.add(create_partial_str(item))

for item in unsats:
    cur_str = create_partial_str(item)
    if cur_str in strs:
        print(item)
    strs.add(cur_str)

print(len(sats))
print(len(unsats))
print(len(strs))

[(1, 1, 1, -1), (1, 1, -1, 1), (1, 1, -1, 1), (1, 1, -1, 1)]
[(1, 1, 1, -1, 1), (1, 1, -1, 1), (1, 1, -1, 1), (1, 1, -1, 1)]
[(1, -1, -1, 1), (1, 1, 1, 1), (1, -1, 1, -1), (1, -1, 1, -1)]
572
770
1339


In [3]:
import torch

print(torch.backends.mps.is_available())



True
